# Stage 4B — Group-Level Comparative Synthesis and Sensitivity

This notebook converts validated Stage 4A dimension-level results into group-level comparative evidence without creating a composite score. It separates current and historical leadership snapshots, summarizes directly comparable longitudinal blocks, preserves unequal persistence opportunities, tests the pre-specified Mayora extended-group sensitivity, and formally assesses whether a single overall portfolio winner is defensible.


## Environment Setup

Import deterministic retrieval, integrity-validation, and tabular-analysis libraries. Lock all Stage 4B inputs to the verified Stage 4A publication commit.


In [1]:
from __future__ import annotations

import hashlib
import os
import shutil
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
REPOSITORY_HEAD = "6ee32d06e296b8e06f1976fb7ba1bc3dcbe5f935"
ANALYSIS_REFERENCE_DATE = "2026-08-21"

FOCAL_GROUPS = [
    "Wings Group",
    "Indofood",
    "Mayora",
    "Unilever Indonesia",
]

STAGE4A_MANIFEST_PATH = "metadata/stage4_output_manifest.csv"

STAGE4A_TRACKED_FILES = [
    "data/analytical/structural_brand_breadth_results.csv",
    "data/analytical/competitive_breadth_results.csv",
    "data/analytical/category_strength_results.csv",
    "data/analytical/category_leadership_results.csv",
    "data/analytical/longitudinal_consistency_results.csv",
    "data/analytical/momentum_results.csv",
    "data/analytical/competitive_persistence_results.csv",
    "data/analytical/dimension_level_summary.csv",
    "metadata/stage4_analysis_validation.csv",
]

SUPPLEMENTAL_LOCKED_FILES = [
    "data/analytical/structural_portfolio_universe.csv",
    "data/analytical/competitive_observation_universe.csv",
]

SUPPLEMENTAL_EXPECTED_SHA256 = {
    "data/analytical/structural_portfolio_universe.csv":
        "08efb815eb0fb7fafbff2b87ad3c2ec5c9481441ff3e15d481ce51315abb89ab",
    "data/analytical/competitive_observation_universe.csv":
        "9996d0d5c8db7587f1d6d84e2ba41118d1161ac9ca7dd98e317b80861fcdd1af",
}

REQUIRED_FILES = [
    *STAGE4A_TRACKED_FILES,
    STAGE4A_MANIFEST_PATH,
    *SUPPLEMENTAL_LOCKED_FILES,
]

OUTPUT_ROOT = Path(
    os.environ.get("FMCG_STAGE4B_OUTPUT_ROOT", "/content/stage4b_outputs")
)
ANALYTICAL_ROOT = OUTPUT_ROOT / "data" / "analytical"
METADATA_ROOT = OUTPUT_ROOT / "metadata"
ANALYTICAL_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_ROOT.mkdir(parents=True, exist_ok=True)


## Locked Committed Input Retrieval

Retrieve every required file from the exact Stage 4A commit. The Colab secret is used only for authenticated GitHub requests and is never written into analytical outputs.


In [2]:
configured_root = os.environ.get("FMCG_STAGE4B_INPUT_ROOT")

if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Run this notebook in Google Colab or set FMCG_STAGE4B_INPUT_ROOT "
            "for local validation."
        ) from exc

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError(
            "The Colab Secret GITHUB_TOKEN is unavailable or access has not "
            "been granted to this notebook."
        )

    INPUT_ROOT = Path("/content/fmcg_stage4b_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for relative_path in REQUIRED_FILES:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        api_url = (
            f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}/contents/"
            f"{encoded_path}?ref={REPOSITORY_HEAD}"
        )
        request = urllib.request.Request(
            api_url,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github.raw+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "fmcg-stage4b-colab",
            },
        )

        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)

        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"GitHub input retrieval failed for {relative_path} "
                f"with HTTP {exc.code}."
            ) from exc

    del github_token
    input_mode = "locked_github_commit"

missing_inputs = [
    relative_path
    for relative_path in REQUIRED_FILES
    if not (INPUT_ROOT / relative_path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing required inputs: {missing_inputs}")

print(f"Input mode: {input_mode}")
print(f"Locked repository head: {REPOSITORY_HEAD}")
print(f"Required files found: {len(REQUIRED_FILES)}/{len(REQUIRED_FILES)}")


Input mode: locked_github_commit
Locked repository head: 6ee32d06e296b8e06f1976fb7ba1bc3dcbe5f935
Required files found: 12/12


## Stage 4A and Supplemental Input Integrity

Verify all Stage 4A tracked outputs against the committed Stage 4A manifest. Independently verify the two supplemental Stage 3 analytical universes used only for the pre-specified ownership sensitivity.


In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv(relative_path: str) -> pd.DataFrame:
    return pd.read_csv(
        INPUT_ROOT / relative_path,
        dtype=str,
        keep_default_na=False,
    )


stage4a_manifest = read_csv(STAGE4A_MANIFEST_PATH)

required_manifest_columns = {
    "file_path",
    "row_count",
    "sha256",
    "description",
    "repository_head",
}
missing_manifest_columns = (
    required_manifest_columns - set(stage4a_manifest.columns)
)
if missing_manifest_columns:
    raise KeyError(
        f"Stage 4A manifest missing columns: {sorted(missing_manifest_columns)}"
    )

if set(stage4a_manifest["file_path"]) != set(STAGE4A_TRACKED_FILES):
    raise ValueError("Stage 4A manifest file scope differs from the locked expectation.")

integrity_rows = []

for row in stage4a_manifest.itertuples(index=False):
    path = INPUT_ROOT / row.file_path
    actual_sha256 = sha256_file(path)
    actual_row_count = len(
        pd.read_csv(path, dtype=str, keep_default_na=False)
    )

    integrity_rows.append(
        {
            "file_path": row.file_path,
            "expected_row_count": int(row.row_count),
            "actual_row_count": actual_row_count,
            "expected_sha256": row.sha256,
            "actual_sha256": actual_sha256,
            "row_count_status": (
                "passed" if actual_row_count == int(row.row_count) else "failed"
            ),
            "sha256_status": (
                "passed" if actual_sha256 == row.sha256 else "failed"
            ),
        }
    )

for relative_path, expected_sha256 in SUPPLEMENTAL_EXPECTED_SHA256.items():
    path = INPUT_ROOT / relative_path
    actual_sha256 = sha256_file(path)
    integrity_rows.append(
        {
            "file_path": relative_path,
            "expected_row_count": "",
            "actual_row_count": len(
                pd.read_csv(path, dtype=str, keep_default_na=False)
            ),
            "expected_sha256": expected_sha256,
            "actual_sha256": actual_sha256,
            "row_count_status": "not_applicable",
            "sha256_status": (
                "passed" if actual_sha256 == expected_sha256 else "failed"
            ),
        }
    )

input_integrity = pd.DataFrame(integrity_rows)

if not input_integrity["sha256_status"].eq("passed").all():
    raise ValueError("At least one locked Stage 4B input failed SHA-256 validation.")

stage4a_rows = input_integrity[
    input_integrity["file_path"].isin(STAGE4A_TRACKED_FILES)
]
if not stage4a_rows["row_count_status"].eq("passed").all():
    raise ValueError("At least one Stage 4A output failed row-count validation.")

print(
    input_integrity[
        ["file_path", "actual_row_count", "row_count_status", "sha256_status"]
    ].to_string(index=False)
)


                                           file_path  actual_row_count row_count_status sha256_status
data/analytical/structural_brand_breadth_results.csv                 4           passed        passed
     data/analytical/competitive_breadth_results.csv                 8           passed        passed
       data/analytical/category_strength_results.csv                75           passed        passed
     data/analytical/category_leadership_results.csv                22           passed        passed
data/analytical/longitudinal_consistency_results.csv                14           passed        passed
                data/analytical/momentum_results.csv                14           passed        passed
 data/analytical/competitive_persistence_results.csv                 4           passed        passed
         data/analytical/dimension_level_summary.csv                11           passed        passed
             metadata/stage4_analysis_validation.csv                26           p

## Analytical Input Loading and Stage Gate

Load the validated Stage 4A results and supplemental universes, check required schemas, and confirm that Stage 4A ended at `PASS_WITH_CAVEAT` with no critical failure.


In [4]:
structural_breadth = read_csv(
    "data/analytical/structural_brand_breadth_results.csv"
)
competitive_breadth = read_csv(
    "data/analytical/competitive_breadth_results.csv"
)
category_strength = read_csv(
    "data/analytical/category_strength_results.csv"
)
category_leadership = read_csv(
    "data/analytical/category_leadership_results.csv"
)
consistency = read_csv(
    "data/analytical/longitudinal_consistency_results.csv"
)
momentum = read_csv(
    "data/analytical/momentum_results.csv"
)
persistence = read_csv(
    "data/analytical/competitive_persistence_results.csv"
)
dimension_summary_4a = read_csv(
    "data/analytical/dimension_level_summary.csv"
)
stage4a_validation = read_csv(
    "metadata/stage4_analysis_validation.csv"
)
structural_universe = read_csv(
    "data/analytical/structural_portfolio_universe.csv"
)
competitive_universe = read_csv(
    "data/analytical/competitive_observation_universe.csv"
)

REQUIRED_COLUMNS = {
    "structural_breadth": {
        "canonical_group",
        "structural_brand_family_count",
        "breadth_rank",
        "leader_status",
    },
    "category_leadership": {
        "source_subcategory_id",
        "canonical_subcategory",
        "reference_period",
        "methodology_cluster",
        "leader_groups",
        "leader_status",
        "category_denominator_status",
    },
    "consistency": {
        "source_label_series_key",
        "canonical_group",
        "canonical_subcategory",
        "source_subcategory_id",
        "methodology_cluster",
        "analysis_periods",
        "comparison_series_count",
        "within_category_consistency_position",
    },
    "momentum": {
        "source_label_series_key",
        "canonical_group",
        "canonical_subcategory",
        "source_subcategory_id",
        "methodology_cluster",
        "analysis_periods",
        "comparison_series_count",
        "within_category_momentum_position",
        "momentum_direction",
    },
    "persistence": {
        "source_subcategory_id",
        "canonical_subcategory",
        "baseline_incumbent_group",
        "incumbent_group_led_all_follow_ups",
        "overtaken_by_other_focal_group",
        "final_leader_groups",
    },
    "dimension_summary_4a": {
        "dimension_id",
        "dimension",
        "current_eligibility",
        "stage4a_execution_status",
    },
    "stage4a_validation": {
        "check_id",
        "result",
        "status",
        "critical_failure",
    },
    "structural_universe": {
        "group",
        "brand_family",
        "relationship_type",
        "current_at_reference_date",
        "strict_control_primary",
        "structural_brand_count_key",
        "structural_brand_breadth_eligible",
        "structural_exclusion_reason",
    },
    "competitive_universe": {
        "source_family",
        "canonical_group",
        "canonical_brand_family",
        "relationship_type",
        "primary_analysis_eligible",
        "sensitivity_analysis_eligible",
        "analysis_scope",
        "category_strength_candidate",
        "category_leadership_value_candidate",
        "consumer_reach_group_comparison_eligible",
    },
}

TABLES = {
    "structural_breadth": structural_breadth,
    "category_leadership": category_leadership,
    "consistency": consistency,
    "momentum": momentum,
    "persistence": persistence,
    "dimension_summary_4a": dimension_summary_4a,
    "stage4a_validation": stage4a_validation,
    "structural_universe": structural_universe,
    "competitive_universe": competitive_universe,
}

for name, required in REQUIRED_COLUMNS.items():
    missing = sorted(required - set(TABLES[name].columns))
    if missing:
        raise KeyError(f"{name} missing required columns: {missing}")

gate = stage4a_validation[
    stage4a_validation["check_id"].eq("S4A026")
]
if len(gate) != 1 or gate.iloc[0]["result"] != "PASS_WITH_CAVEAT":
    raise RuntimeError("Stage 4A final gate is not the expected PASS_WITH_CAVEAT.")

if stage4a_validation["critical_failure"].str.lower().eq("yes").any():
    raise RuntimeError("Stage 4A contains a critical validation failure.")

print("Stage 4A gate: PASS_WITH_CAVEAT")
print("Input schemas: passed")


Stage 4A gate: PASS_WITH_CAVEAT
Input schemas: passed


## Current and Latest-Historical Focal Leadership Snapshots

Create two non-overlapping category snapshots: the documented 2026 methodology cluster and the latest eligible historical observation for each category. Count only complete focal-group leadership denominators; the resulting rate is a share of the selected eligible focal categories, not market share.


In [5]:
def split_group_field(value: str) -> list[str]:
    return [
        item.strip()
        for item in str(value).split("|")
        if item.strip()
    ]


def summarize_leadership_snapshot(
    frame: pd.DataFrame,
    snapshot_type: str,
    snapshot_definition: str,
) -> pd.DataFrame:
    if frame.empty:
        raise ValueError(f"Leadership snapshot {snapshot_type} is empty.")

    if not frame["category_denominator_status"].eq("complete_observed").all():
        raise ValueError(
            f"Leadership snapshot {snapshot_type} contains incomplete denominators."
        )

    eligible_count = len(frame)
    rows = []

    for group in FOCAL_GROUPS:
        lead_count = sum(
            group in split_group_field(value)
            for value in frame["leader_groups"]
        )
        rows.append(
            {
                "snapshot_type": snapshot_type,
                "snapshot_definition": snapshot_definition,
                "canonical_group": group,
                "eligible_focal_category_count": eligible_count,
                "focal_category_lead_count": int(lead_count),
                "focal_category_leadership_rate": (
                    float(lead_count / eligible_count)
                    if eligible_count > 0
                    else np.nan
                ),
                "rate_semantics": (
                    "Share of selected eligible complete focal-category events; "
                    "not market share and not a full-market leadership rate."
                ),
            }
        )

    return pd.DataFrame(rows)


leadership_work = category_leadership.copy()
leadership_work["reference_period_int"] = pd.to_numeric(
    leadership_work["reference_period"],
    errors="raise",
).astype(int)

current_snapshot = leadership_work[
    leadership_work["methodology_cluster"].eq("top_brand_current_documented")
    & leadership_work["reference_period_int"].eq(2026)
].copy()

historical = leadership_work[
    leadership_work["methodology_cluster"].eq("top_brand_historical_unverified")
].copy()

latest_historical_indexes = (
    historical.groupby("source_subcategory_id")["reference_period_int"].idxmax()
)
latest_historical_snapshot = historical.loc[
    latest_historical_indexes
].copy()

leadership_snapshot_summary = pd.concat(
    [
        summarize_leadership_snapshot(
            current_snapshot,
            "current_documented_2026",
            "Documented 2026 methodology cluster only.",
        ),
        summarize_leadership_snapshot(
            latest_historical_snapshot,
            "latest_eligible_historical",
            (
                "Latest complete historical focal-leadership period within each "
                "eligible category; no bridge to 2026."
            ),
        ),
    ],
    ignore_index=True,
)

print(leadership_snapshot_summary.to_string(index=False))


             snapshot_type                                                                                  snapshot_definition    canonical_group  eligible_focal_category_count  focal_category_lead_count  focal_category_leadership_rate                                                                                                     rate_semantics
   current_documented_2026                                                            Documented 2026 methodology cluster only.        Wings Group                              5                          2                             0.4 Share of selected eligible complete focal-category events; not market share and not a full-market leadership rate.
   current_documented_2026                                                            Documented 2026 methodology cluster only.           Indofood                              5                          0                             0.0 Share of selected eligible complete focal-category events; 

## Comparable Longitudinal Blocks

Collapse consistency and momentum into directly comparable category-period blocks only where at least two source-label series share the same category, methodology cluster, and analyzed period block. Stability and momentum remain separate concepts and are not averaged into one score.


In [6]:
BLOCK_KEYS = [
    "canonical_subcategory",
    "source_subcategory_id",
    "methodology_cluster",
    "analysis_periods",
]


def build_block_summary(
    frame: pd.DataFrame,
    dimension: str,
    position_column: str,
    interpretation: str,
) -> pd.DataFrame:
    work = frame.copy()
    work["comparison_series_count_int"] = pd.to_numeric(
        work["comparison_series_count"],
        errors="raise",
    ).astype(int)
    work["position_int"] = pd.to_numeric(
        work[position_column],
        errors="raise",
    ).astype(int)

    work = work[
        work["comparison_series_count_int"].ge(2)
    ].copy()

    rows = []

    for key_values, block in work.groupby(
        BLOCK_KEYS,
        dropna=False,
        sort=True,
    ):
        key = dict(zip(BLOCK_KEYS, key_values))

        leaders = block[
            block["position_int"].eq(block["position_int"].min())
        ].copy()

        leader_groups = sorted(leaders["canonical_group"].unique())
        leader_series = " | ".join(
            sorted(leaders["source_label_series_key"].unique())
        )

        rows.append(
            {
                "analysis_dimension": dimension,
                **key,
                "comparison_series_count": int(len(block)),
                "comparison_group_count": int(
                    block["canonical_group"].nunique()
                ),
                "leader_group_count": len(leader_groups),
                "leader_groups": " | ".join(leader_groups),
                "leader_series_keys": leader_series,
                "block_interpretation": interpretation,
            }
        )

    return pd.DataFrame(rows)


consistency_blocks = build_block_summary(
    consistency,
    "observed_stability",
    "within_category_consistency_position",
    (
        "Lower mean absolute period-to-period TBI change indicates greater "
        "observed stability within this block only; stability is not strength."
    ),
)

momentum_blocks = build_block_summary(
    momentum,
    "momentum",
    "within_category_momentum_position",
    (
        "Higher net TBI change indicates stronger momentum within this directly "
        "comparable block only; no cross-category magnitude averaging is used."
    ),
)

comparable_longitudinal_block_summary = pd.concat(
    [consistency_blocks, momentum_blocks],
    ignore_index=True,
)

print(comparable_longitudinal_block_summary.to_string(index=False))


analysis_dimension       canonical_subcategory source_subcategory_id             methodology_cluster    analysis_periods  comparison_series_count  comparison_group_count  leader_group_count      leader_groups                                                                         leader_series_keys                                                                                                                         block_interpretation
observed_stability        Antiseptic Bath Soap                   274 top_brand_historical_unverified 2022;2023;2024;2025                        2                       2                   1        Wings Group          AU019|NUVO|274|TBI|percent_index_points|Indonesia|top_brand_historical_unverified      Lower mean absolute period-to-period TBI change indicates greater observed stability within this block only; stability is not strength.
observed_stability      Bagged Instant Noodles                    30 top_brand_historical_unverified 2022;2023;2024;20

## Competitive Persistence by Observed Role

Summarize incumbent retention opportunities separately from challenger takeovers. Do not rank groups with no observed baseline-incumbent opportunity as zero-persistence performers.


In [7]:
persistence_rows = []

for group in FOCAL_GROUPS:
    incumbent_rows = persistence[
        persistence["baseline_incumbent_group"].eq(group)
    ].copy()

    baseline_opportunities = len(incumbent_rows)

    if baseline_opportunities > 0:
        retained_count = int(
            incumbent_rows[
                "incumbent_group_led_all_follow_ups"
            ].eq("yes").sum()
        )
        lost_count = int(baseline_opportunities - retained_count)
        retention_rate = retained_count / baseline_opportunities
        opportunity_status = "observed_baseline_incumbent_opportunity"
    else:
        retained_count = 0
        lost_count = 0
        retention_rate = np.nan
        opportunity_status = "not_observed_as_baseline_incumbent"

    challenger_takeovers = 0
    for row in persistence[
        persistence["overtaken_by_other_focal_group"].eq("yes")
    ].itertuples(index=False):
        final_groups = split_group_field(row.final_leader_groups)
        if (
            group in final_groups
            and group != row.baseline_incumbent_group
        ):
            challenger_takeovers += 1

    persistence_rows.append(
        {
            "canonical_group": group,
            "baseline_incumbent_opportunity_count": baseline_opportunities,
            "retained_incumbent_all_followups_count": retained_count,
            "lost_incumbent_count": lost_count,
            "incumbent_retention_rate": retention_rate,
            "persistence_opportunity_status": opportunity_status,
            "successful_challenger_takeover_count": challenger_takeovers,
            "persistence_semantics": (
                "Incumbent retention and challenger takeover are separate "
                "observed roles; unavailable incumbent opportunity is not zero."
            ),
        }
    )

persistence_group_summary = pd.DataFrame(persistence_rows)

print(persistence_group_summary.to_string(index=False))


   canonical_group  baseline_incumbent_opportunity_count  retained_incumbent_all_followups_count  lost_incumbent_count  incumbent_retention_rate          persistence_opportunity_status  successful_challenger_takeover_count                                                                                                   persistence_semantics
       Wings Group                                     0                                       0                     0                       NaN      not_observed_as_baseline_incumbent                                     1 Incumbent retention and challenger takeover are separate observed roles; unavailable incumbent opportunity is not zero.
          Indofood                                     1                                       0                     1                       0.0 observed_baseline_incumbent_opportunity                                     0 Incumbent retention and challenger takeover are separate observed roles; unavailable incumb

## Pre-Specified Ownership and Temporal Sensitivity

Test the extended Mayora scenario by adding only the already-identified Le Minerale related-party affiliate. Separately verify that its public Brand Footprint facts remain context-only and test whether the current and latest-historical focal leadership category patterns are stable.


In [8]:
primary_counts = structural_breadth.copy()
primary_counts["structural_brand_family_count_int"] = pd.to_numeric(
    primary_counts["structural_brand_family_count"],
    errors="raise",
).astype(int)
primary_counts["breadth_rank_int"] = pd.to_numeric(
    primary_counts["breadth_rank"],
    errors="raise",
).astype(int)

mayora_primary_row = primary_counts[
    primary_counts["canonical_group"].eq("Mayora")
]
if len(mayora_primary_row) != 1:
    raise ValueError("Primary Mayora structural breadth row is not unique.")

mayora_primary_count = int(
    mayora_primary_row.iloc[0]["structural_brand_family_count_int"]
)
mayora_primary_rank = int(
    mayora_primary_row.iloc[0]["breadth_rank_int"]
)

le_minerale_structural = structural_universe[
    structural_universe["group"].eq("Mayora")
    & structural_universe["brand_family"].eq("Le Minerale")
    & structural_universe["current_at_reference_date"].eq("yes")
    & structural_universe["strict_control_primary"].eq("no")
    & structural_universe["relationship_type"].eq("related_group_affiliate")
].copy()

le_minerale_additional_keys = sorted(
    set(
        value
        for value in le_minerale_structural["structural_brand_count_key"]
        if value
    )
)

if len(le_minerale_additional_keys) != 1:
    raise ValueError(
        "Expected exactly one distinct Le Minerale structural sensitivity key."
    )

extended_counts = primary_counts[
    ["canonical_group", "structural_brand_family_count_int"]
].copy()
extended_counts["sensitivity_brand_family_count"] = (
    extended_counts["structural_brand_family_count_int"]
)

extended_counts.loc[
    extended_counts["canonical_group"].eq("Mayora"),
    "sensitivity_brand_family_count",
] = mayora_primary_count + len(le_minerale_additional_keys)

extended_counts["sensitivity_rank"] = (
    extended_counts["sensitivity_brand_family_count"]
    .rank(method="min", ascending=False)
    .astype(int)
)

mayora_extended_row = extended_counts[
    extended_counts["canonical_group"].eq("Mayora")
].iloc[0]
mayora_extended_count = int(
    mayora_extended_row["sensitivity_brand_family_count"]
)
mayora_extended_rank = int(
    mayora_extended_row["sensitivity_rank"]
)

primary_leaders = sorted(
    primary_counts.loc[
        primary_counts["breadth_rank_int"].eq(1),
        "canonical_group",
    ].tolist()
)
extended_leaders = sorted(
    extended_counts.loc[
        extended_counts["sensitivity_rank"].eq(1),
        "canonical_group",
    ].tolist()
)

le_minerale_competitive = competitive_universe[
    competitive_universe["canonical_brand_family"].eq("Le Minerale")
    & competitive_universe["sensitivity_analysis_eligible"].eq("yes")
].copy()

if len(le_minerale_competitive) != 2:
    raise ValueError(
        "Expected exactly two Le Minerale competitive sensitivity observations."
    )

le_minerale_performance_candidates = int(
    (
        le_minerale_competitive["category_strength_candidate"].eq("yes")
        | le_minerale_competitive[
            "category_leadership_value_candidate"
        ].eq("yes")
        | le_minerale_competitive[
            "consumer_reach_group_comparison_eligible"
        ].eq("yes")
    ).sum()
)

current_pattern = (
    current_snapshot.set_index("source_subcategory_id")["leader_groups"]
    .sort_index()
)
historical_pattern = (
    latest_historical_snapshot.set_index("source_subcategory_id")["leader_groups"]
    .sort_index()
)

same_category_set = (
    set(current_pattern.index) == set(historical_pattern.index)
)
same_leader_pattern = (
    same_category_set
    and current_pattern.equals(
        historical_pattern.reindex(current_pattern.index)
    )
)

sensitivity_robustness_results = pd.DataFrame(
    [
        {
            "test_id": "SENS01",
            "test_type": "ownership_scope",
            "test": "Mayora structural brand breadth with Le Minerale added",
            "primary_result": (
                f"Mayora brand-family count={mayora_primary_count}; "
                f"rank={mayora_primary_rank}; leaders={' | '.join(primary_leaders)}"
            ),
            "sensitivity_result": (
                f"Mayora brand-family count={mayora_extended_count}; "
                f"rank={mayora_extended_rank}; leaders={' | '.join(extended_leaders)}"
            ),
            "material_change": (
                "yes"
                if (
                    mayora_primary_rank != mayora_extended_rank
                    or primary_leaders != extended_leaders
                )
                else "no"
            ),
            "interpretation": (
                "Extended-group sensitivity changes Mayora breadth count only; "
                "primary ownership attribution remains unchanged."
            ),
        },
        {
            "test_id": "SENS02",
            "test_type": "ownership_scope",
            "test": "Le Minerale competitive evidence eligibility",
            "primary_result": "Le Minerale excluded from strict-control primary performance.",
            "sensitivity_result": (
                f"{len(le_minerale_competitive)} sensitivity observations; "
                f"{le_minerale_performance_candidates} eligible quantitative "
                "group-performance candidates."
            ),
            "material_change": (
                "yes" if le_minerale_performance_candidates > 0 else "no"
            ),
            "interpretation": (
                "The Le Minerale observations remain Brand Footprint context-only "
                "and cannot alter Top Brand leadership, consistency, momentum, "
                "or persistence results."
            ),
        },
        {
            "test_id": "ROB01",
            "test_type": "temporal_robustness",
            "test": "Current versus latest-historical focal leadership pattern",
            "primary_result": (
                f"Current complete categories={len(current_pattern)}"
            ),
            "sensitivity_result": (
                f"Latest historical complete categories={len(historical_pattern)}; "
                f"same category set={same_category_set}; "
                f"same category leader pattern={same_leader_pattern}"
            ),
            "material_change": "no" if same_leader_pattern else "yes",
            "interpretation": (
                "This check compares category-level leader identity only and does "
                "not treat 2025-to-2026 TBI movement as longitudinally comparable."
            ),
        },
    ]
)

print(sensitivity_robustness_results.to_string(index=False))


test_id           test_type                                                      test                                                primary_result                                                                                 sensitivity_result material_change                                                                                                                                         interpretation
 SENS01     ownership_scope    Mayora structural brand breadth with Le Minerale added        Mayora brand-family count=22; rank=4; leaders=Indofood                                             Mayora brand-family count=23; rank=4; leaders=Indofood              no                                         Extended-group sensitivity changes Mayora breadth count only; primary ownership attribution remains unchanged.
 SENS02     ownership_scope              Le Minerale competitive evidence eligibility Le Minerale excluded from strict-control primary performance.                  2 sensi

## Group-Dimension Evidence Matrix and Dimension Signals

Assemble group-level evidence without normalizing or weighting incompatible dimensions. For longitudinal dimensions, count comparable block-leader events only; for persistence, preserve the role-specific opportunity denominator.


In [9]:
def snapshot_rows(snapshot_type: str) -> pd.DataFrame:
    return leadership_snapshot_summary[
        leadership_snapshot_summary["snapshot_type"].eq(snapshot_type)
    ].copy()


current_summary = snapshot_rows("current_documented_2026")
historical_summary = snapshot_rows("latest_eligible_historical")


def block_leader_counts(dimension: str) -> dict[str, int]:
    frame = comparable_longitudinal_block_summary[
        comparable_longitudinal_block_summary["analysis_dimension"].eq(dimension)
    ]
    counts = {group: 0 for group in FOCAL_GROUPS}

    for value in frame["leader_groups"]:
        for group in split_group_field(value):
            if group in counts:
                counts[group] += 1

    return counts


consistency_leader_counts = block_leader_counts("observed_stability")
momentum_leader_counts = block_leader_counts("momentum")

consistency_block_count = int(
    comparable_longitudinal_block_summary[
        "analysis_dimension"
    ].eq("observed_stability").sum()
)
momentum_block_count = int(
    comparable_longitudinal_block_summary[
        "analysis_dimension"
    ].eq("momentum").sum()
)

evidence_rows = []

for group in FOCAL_GROUPS:
    structural_row = primary_counts[
        primary_counts["canonical_group"].eq(group)
    ].iloc[0]

    current_row = current_summary[
        current_summary["canonical_group"].eq(group)
    ].iloc[0]

    historical_row = historical_summary[
        historical_summary["canonical_group"].eq(group)
    ].iloc[0]

    persistence_row = persistence_group_summary[
        persistence_group_summary["canonical_group"].eq(group)
    ].iloc[0]

    evidence_rows.extend(
        [
            {
                "canonical_group": group,
                "dimension": "structural_brand_breadth",
                "evidence_value": int(
                    structural_row["structural_brand_family_count_int"]
                ),
                "eligible_denominator": "",
                "relative_position_or_rate": int(
                    structural_row["breadth_rank_int"]
                ),
                "evidence_status": "eligible_with_caveat",
                "semantics": (
                    "Distinct current strict-control canonical brand families; "
                    "relative position is breadth rank."
                ),
            },
            {
                "canonical_group": group,
                "dimension": "current_focal_category_leadership",
                "evidence_value": int(current_row["focal_category_lead_count"]),
                "eligible_denominator": int(
                    current_row["eligible_focal_category_count"]
                ),
                "relative_position_or_rate": float(
                    current_row["focal_category_leadership_rate"]
                ),
                "evidence_status": "eligible_with_caveat",
                "semantics": (
                    "Complete selected 2026 focal-category leadership events; "
                    "rate is not market share."
                ),
            },
            {
                "canonical_group": group,
                "dimension": "latest_historical_focal_category_leadership",
                "evidence_value": int(historical_row["focal_category_lead_count"]),
                "eligible_denominator": int(
                    historical_row["eligible_focal_category_count"]
                ),
                "relative_position_or_rate": float(
                    historical_row["focal_category_leadership_rate"]
                ),
                "evidence_status": "eligible_with_caveat",
                "semantics": (
                    "Latest complete historical focal leadership event per "
                    "eligible category; no 2026 bridge."
                ),
            },
            {
                "canonical_group": group,
                "dimension": "observed_stability_block_leadership",
                "evidence_value": consistency_leader_counts[group],
                "eligible_denominator": consistency_block_count,
                "relative_position_or_rate": (
                    consistency_leader_counts[group] / consistency_block_count
                    if consistency_block_count
                    else np.nan
                ),
                "evidence_status": "eligible_with_caveat",
                "semantics": (
                    "Count of directly comparable longitudinal blocks where the "
                    "group contains the most stable source-label series; "
                    "stability is not strength."
                ),
            },
            {
                "canonical_group": group,
                "dimension": "momentum_block_leadership",
                "evidence_value": momentum_leader_counts[group],
                "eligible_denominator": momentum_block_count,
                "relative_position_or_rate": (
                    momentum_leader_counts[group] / momentum_block_count
                    if momentum_block_count
                    else np.nan
                ),
                "evidence_status": "eligible_with_caveat",
                "semantics": (
                    "Count of directly comparable longitudinal blocks where the "
                    "group contains the highest net-TBI-change series."
                ),
            },
            {
                "canonical_group": group,
                "dimension": "incumbent_persistence",
                "evidence_value": int(
                    persistence_row["retained_incumbent_all_followups_count"]
                ),
                "eligible_denominator": (
                    int(persistence_row["baseline_incumbent_opportunity_count"])
                    if int(
                        persistence_row["baseline_incumbent_opportunity_count"]
                    ) > 0
                    else ""
                ),
                "relative_position_or_rate": (
                    float(persistence_row["incumbent_retention_rate"])
                    if pd.notna(persistence_row["incumbent_retention_rate"])
                    else np.nan
                ),
                "evidence_status": persistence_row[
                    "persistence_opportunity_status"
                ],
                "semantics": (
                    "Incumbent retention conditional on observed baseline-incumbent "
                    "opportunities; no opportunity is not a zero."
                ),
            },
            {
                "canonical_group": group,
                "dimension": "successful_challenger_takeover",
                "evidence_value": int(
                    persistence_row["successful_challenger_takeover_count"]
                ),
                "eligible_denominator": len(persistence),
                "relative_position_or_rate": "",
                "evidence_status": "descriptive_role_specific",
                "semantics": (
                    "Count of eligible persistence records ending with this group "
                    "leading after another focal group was the baseline incumbent."
                ),
            },
        ]
    )

group_dimension_evidence_matrix = pd.DataFrame(evidence_rows)


def leader_groups_from_values(
    frame: pd.DataFrame,
    value_column: str,
) -> list[str]:
    values = pd.to_numeric(frame[value_column], errors="raise")
    max_value = values.max()
    return sorted(
        frame.loc[values.eq(max_value), "canonical_group"].tolist()
    )


structural_leaders = sorted(
    primary_counts.loc[
        primary_counts["breadth_rank_int"].eq(1),
        "canonical_group",
    ].tolist()
)

current_leaders = leader_groups_from_values(
    current_summary,
    "focal_category_lead_count",
)

historical_leaders = leader_groups_from_values(
    historical_summary,
    "focal_category_lead_count",
)

consistency_max = max(consistency_leader_counts.values())
consistency_leaders = sorted(
    group
    for group, count in consistency_leader_counts.items()
    if count == consistency_max
)

momentum_max = max(momentum_leader_counts.values())
momentum_leaders = sorted(
    group
    for group, count in momentum_leader_counts.items()
    if count == momentum_max
)

retention_observed = persistence_group_summary[
    persistence_group_summary["baseline_incumbent_opportunity_count"].gt(0)
].copy()

retention_signal = "; ".join(
    (
        f"{row.canonical_group}: "
        f"{row.retained_incumbent_all_followups_count}/"
        f"{row.baseline_incumbent_opportunity_count}"
    )
    for row in retention_observed.itertuples(index=False)
)

takeover_signal = "; ".join(
    (
        f"{row.canonical_group}: "
        f"{row.successful_challenger_takeover_count}"
    )
    for row in persistence_group_summary[
        persistence_group_summary["successful_challenger_takeover_count"].gt(0)
    ].itertuples(index=False)
)

dimension_leader_summary = pd.DataFrame(
    [
        {
            "perspective": "structural_brand_breadth",
            "leader_status": "rankable_with_caveat",
            "leader_groups": " | ".join(structural_leaders),
            "evidence_basis": "Current strict-control canonical brand-family count.",
            "comparability_caveat": (
                "Structural category breadth remains unavailable because category "
                "mapping is incomplete."
            ),
        },
        {
            "perspective": "current_focal_category_leadership",
            "leader_status": "rankable_with_caveat",
            "leader_groups": " | ".join(current_leaders),
            "evidence_basis": (
                f"{len(current_snapshot)} complete selected 2026 focal categories."
            ),
            "comparability_caveat": (
                "Focal-group leadership only; not full-market leadership."
            ),
        },
        {
            "perspective": "latest_historical_focal_category_leadership",
            "leader_status": "rankable_with_caveat",
            "leader_groups": " | ".join(historical_leaders),
            "evidence_basis": (
                f"{len(latest_historical_snapshot)} latest complete historical "
                "category snapshots."
            ),
            "comparability_caveat": (
                "Historical methodology is not independently verified and is not "
                "bridged to 2026."
            ),
        },
        {
            "perspective": "observed_stability",
            "leader_status": "rankable_with_caveat",
            "leader_groups": " | ".join(consistency_leaders),
            "evidence_basis": (
                f"{consistency_block_count} directly comparable longitudinal blocks."
            ),
            "comparability_caveat": (
                "Stability is not category strength; singleton blocks are excluded."
            ),
        },
        {
            "perspective": "momentum",
            "leader_status": "rankable_with_caveat",
            "leader_groups": " | ".join(momentum_leaders),
            "evidence_basis": (
                f"{momentum_block_count} directly comparable longitudinal blocks."
            ),
            "comparability_caveat": (
                "Net TBI changes are compared only within category-period blocks."
            ),
        },
        {
            "perspective": "competitive_persistence_incumbent_retention",
            "leader_status": "descriptive_signal_only",
            "leader_groups": "",
            "evidence_basis": retention_signal,
            "comparability_caveat": (
                "Baseline-incumbent opportunities are unequal across groups, so "
                "a cross-group persistence rank is not produced."
            ),
        },
        {
            "perspective": "competitive_persistence_challenger_takeover",
            "leader_status": "descriptive_signal_only",
            "leader_groups": "",
            "evidence_basis": takeover_signal if takeover_signal else "No takeover observed.",
            "comparability_caveat": (
                "Takeover opportunities are event-specific and not a portfolio-wide rate."
            ),
        },
        {
            "perspective": "consumer_reach",
            "leader_status": "not_comparable",
            "leader_groups": "",
            "evidence_basis": "No comparable source-native CRP universe across focal groups.",
            "comparability_caveat": "Brand Footprint public facts remain context-only.",
        },
        {
            "perspective": "portfolio_concentration",
            "leader_status": "not_eligible",
            "leader_groups": "",
            "evidence_basis": "Performance-weighted HHI remains prohibited.",
            "comparability_caveat": (
                "Current cross-category evidence is not an additive portfolio distribution."
            ),
        },
    ]
)

print(group_dimension_evidence_matrix.to_string(index=False))
print()
print(dimension_leader_summary.to_string(index=False))


   canonical_group                                   dimension  evidence_value eligible_denominator relative_position_or_rate                         evidence_status                                                                                                                                 semantics
       Wings Group                    structural_brand_breadth              37                                              2                    eligible_with_caveat                                              Distinct current strict-control canonical brand families; relative position is breadth rank.
       Wings Group           current_focal_category_leadership               2                    5                       0.4                    eligible_with_caveat                                                        Complete selected 2026 focal-category leadership events; rate is not market share.
       Wings Group latest_historical_focal_category_leadership               2          

## Overall-Winner Defensibility Gates

Apply explicit synthesis gates before any overall-winner statement. Missing core dimensions, incompatible scales, selective coverage, and divergent dimension leaders are treated as substantive constraints rather than filled with arbitrary weights or zeroes.


In [10]:
rankable = dimension_leader_summary[
    dimension_leader_summary["leader_status"].eq("rankable_with_caveat")
].copy()

rankable_leader_sets = [
    tuple(split_group_field(value))
    for value in rankable["leader_groups"]
]

unique_rankable_leader_sets = sorted(set(rankable_leader_sets))
leader_convergence = len(unique_rankable_leader_sets) == 1

consumer_reach_available = (
    dimension_leader_summary.loc[
        dimension_leader_summary["perspective"].eq("consumer_reach"),
        "leader_status",
    ].iloc[0]
    != "not_comparable"
)

concentration_available = (
    dimension_leader_summary.loc[
        dimension_leader_summary["perspective"].eq("portfolio_concentration"),
        "leader_status",
    ].iloc[0]
    != "not_eligible"
)

structural_category_available = (
    dimension_summary_4a.loc[
        dimension_summary_4a["dimension_id"].eq("DIM01B"),
        "stage4a_execution_status",
    ].iloc[0]
    != "not_computed_not_eligible"
)

selective_coverage_remains = (
    competitive_breadth["performance_ranking_status"]
    .eq("not_ranked_due_selective_public_coverage")
    .all()
)

ownership_sensitivity_material = (
    sensitivity_robustness_results.loc[
        sensitivity_robustness_results["test_id"].isin(["SENS01", "SENS02"]),
        "material_change",
    ].eq("yes").any()
)

defensibility_rows = [
    {
        "gate_id": "OWG01",
        "gate": "Structural breadth coverage",
        "status": "partial",
        "evidence": (
            "Brand-family breadth is available, but structural category breadth "
            f"available={structural_category_available}."
        ),
        "implication": "Breadth is only partially characterized.",
    },
    {
        "gate_id": "OWG02",
        "gate": "Consumer reach comparability",
        "status": "passed" if consumer_reach_available else "failed",
        "evidence": (
            "Comparable consumer reach is available."
            if consumer_reach_available
            else "No comparable source-native CRP universe across all focal groups."
        ),
        "implication": "A core success dimension is missing." if not consumer_reach_available else "",
    },
    {
        "gate_id": "OWG03",
        "gate": "Category leadership evidence",
        "status": "passed_with_caveat",
        "evidence": (
            f"{len(current_snapshot)} current and "
            f"{len(latest_historical_snapshot)} latest-historical complete "
            "focal-category snapshots."
        ),
        "implication": "Usable only as focal-group leadership evidence.",
    },
    {
        "gate_id": "OWG04",
        "gate": "Longitudinal comparability",
        "status": "passed_with_caveat",
        "evidence": (
            f"{consistency_block_count} stability blocks and "
            f"{momentum_block_count} momentum blocks are directly comparable."
        ),
        "implication": "No cross-category magnitude averaging is permitted.",
    },
    {
        "gate_id": "OWG05",
        "gate": "Competitive persistence breadth",
        "status": "partial",
        "evidence": (
            f"{len(persistence)} eligible persistence records with unequal "
            "baseline-incumbent opportunities."
        ),
        "implication": "Role-specific signals are valid; a portfolio-wide persistence rank is not.",
    },
    {
        "gate_id": "OWG06",
        "gate": "Portfolio concentration comparability",
        "status": "passed" if concentration_available else "failed",
        "evidence": (
            "Comparable concentration measure is available."
            if concentration_available
            else "Performance-weighted HHI remains ineligible."
        ),
        "implication": "A core diversification/concentration dimension is missing." if not concentration_available else "",
    },
    {
        "gate_id": "OWG07",
        "gate": "Coverage neutrality",
        "status": "failed" if selective_coverage_remains else "passed",
        "evidence": (
            "Public competitive coverage remains selective and is not ranked as performance."
            if selective_coverage_remains
            else "Competitive coverage is sufficiently neutral."
        ),
        "implication": "Observable evidence cannot represent the entire structural portfolio.",
    },
    {
        "gate_id": "OWG08",
        "gate": "Cross-dimension leader convergence",
        "status": "passed" if leader_convergence else "failed",
        "evidence": (
            f"Rankable leader sets={unique_rankable_leader_sets}"
        ),
        "implication": (
            "Available dimensions identify different leaders."
            if not leader_convergence
            else "Available rankable dimensions identify the same leader set."
        ),
    },
    {
        "gate_id": "OWG09",
        "gate": "Scale and weighting compatibility",
        "status": "failed",
        "evidence": (
            "Structural counts, focal-category events, stability, momentum, and "
            "persistence have different units and opportunity structures."
        ),
        "implication": "No defensible composite weighting or common scale has been established.",
    },
    {
        "gate_id": "OWG10",
        "gate": "Pre-specified ownership sensitivity",
        "status": "failed" if ownership_sensitivity_material else "passed",
        "evidence": (
            "Le Minerale sensitivity materially changes primary conclusions."
            if ownership_sensitivity_material
            else "Le Minerale sensitivity does not change structural rank or eligible quantitative performance results."
        ),
        "implication": (
            "Primary conclusions are ownership-sensitive."
            if ownership_sensitivity_material
            else "Primary conclusions are stable to the pre-specified Mayora extended-group sensitivity."
        ),
    },
]

overall_winner_defensibility = pd.DataFrame(defensibility_rows)

blocking_statuses = overall_winner_defensibility["status"].isin(["failed"])
overall_winner_defensible = not blocking_statuses.any()

final_row = pd.DataFrame(
    [
        {
            "gate_id": "OWG11",
            "gate": "Overall winner conclusion",
            "status": "defensible" if overall_winner_defensible else "not_defensible",
            "evidence": (
                "All blocking gates passed."
                if overall_winner_defensible
                else (
                    "At least one mandatory synthesis gate failed; current evidence "
                    "supports dimension-level leaders rather than one overall winner."
                )
            ),
            "implication": (
                "An overall winner may be assessed with transparent pre-specified synthesis."
                if overall_winner_defensible
                else "Report multiple dimension leaders and explicit evidence limits."
            ),
        }
    ]
)

overall_winner_defensibility = pd.concat(
    [overall_winner_defensibility, final_row],
    ignore_index=True,
)

print(overall_winner_defensibility.to_string(index=False))


gate_id                                  gate             status                                                                                                                        evidence                                                                            implication
  OWG01           Structural breadth coverage            partial                                             Brand-family breadth is available, but structural category breadth available=False.                                               Breadth is only partially characterized.
  OWG02          Consumer reach comparability             failed                                                               No comparable source-native CRP universe across all focal groups.                                                   A core success dimension is missing.
  OWG03          Category leadership evidence passed_with_caveat                                                            5 current and 5 latest-historical co

## Stage 4B Validation

Validate source integrity, snapshot denominators, comparable-block construction, persistence opportunity treatment, sensitivity isolation, and the prohibition on composite scoring before outputs are written.


In [11]:
validation_rows = []

def add_check(
    check_id,
    area,
    description,
    condition,
    result,
    treatment,
    caveat=False,
    critical=True,
):
    condition = bool(condition)
    status = (
        "passed_with_caveat"
        if condition and caveat
        else "passed"
        if condition
        else "failed"
    )
    validation_rows.append(
        {
            "check_id": check_id,
            "validation_area": area,
            "check_description": description,
            "result": str(result),
            "status": status,
            "critical_failure": "yes" if (not condition and critical) else "no",
            "required_treatment": treatment,
        }
    )


add_check(
    "S4B001",
    "input_integrity",
    "All locked Stage 4A and supplemental inputs pass SHA-256 validation.",
    input_integrity["sha256_status"].eq("passed").all(),
    f"{len(input_integrity)}/{len(input_integrity)} inputs passed SHA-256",
    "Stop if any locked input differs.",
)

add_check(
    "S4B002",
    "prior_stage_gate",
    "Stage 4A final gate is PASS_WITH_CAVEAT with no critical failure.",
    (
        gate.iloc[0]["result"] == "PASS_WITH_CAVEAT"
        and not stage4a_validation["critical_failure"].str.lower().eq("yes").any()
    ),
    "PASS_WITH_CAVEAT",
    "Do not synthesize a failed prior stage.",
    caveat=True,
)

add_check(
    "S4B003",
    "current_leadership_snapshot",
    "Current focal leadership uses only documented 2026 complete denominators.",
    (
        len(current_snapshot) == 5
        and current_snapshot["category_denominator_status"].eq("complete_observed").all()
        and current_snapshot["methodology_cluster"].eq("top_brand_current_documented").all()
    ),
    f"{len(current_snapshot)} current complete categories",
    "Do not mix historical methodology into the current snapshot.",
)

add_check(
    "S4B004",
    "historical_leadership_snapshot",
    "Latest historical focal leadership contains one complete snapshot per eligible category.",
    (
        len(latest_historical_snapshot) == 5
        and latest_historical_snapshot["source_subcategory_id"].is_unique
        and latest_historical_snapshot["category_denominator_status"].eq("complete_observed").all()
    ),
    f"{len(latest_historical_snapshot)} historical category snapshots",
    "Use only the latest complete historical period per category.",
    caveat=True,
)

add_check(
    "S4B005",
    "leadership_semantics",
    "Leadership snapshot rates are explicitly non-market-share selected-category rates.",
    leadership_snapshot_summary["rate_semantics"].str.contains(
        "not market share", case=False, regex=False
    ).all(),
    "Selected-category rate semantics retained",
    "Never relabel focal-category leadership rate as market share.",
)

add_check(
    "S4B006",
    "consistency_blocks",
    "Observed stability synthesis uses only multi-series directly comparable blocks.",
    (
        consistency_block_count == 5
        and consistency_blocks["comparison_series_count"].ge(2).all()
    ),
    f"{consistency_block_count} comparable stability blocks",
    "Exclude singleton longitudinal blocks.",
    caveat=True,
)

add_check(
    "S4B007",
    "momentum_blocks",
    "Momentum synthesis uses only multi-series directly comparable blocks.",
    (
        momentum_block_count == 5
        and momentum_blocks["comparison_series_count"].ge(2).all()
    ),
    f"{momentum_block_count} comparable momentum blocks",
    "Exclude singleton longitudinal blocks.",
    caveat=True,
)

add_check(
    "S4B008",
    "longitudinal_boundary",
    "No comparable longitudinal block includes 2026.",
    ~comparable_longitudinal_block_summary["analysis_periods"].str.contains(
        "2026", regex=False
    ).any(),
    "No 2025-2026 longitudinal bridge",
    "Keep methodology clusters separate.",
)

add_check(
    "S4B009",
    "persistence_opportunities",
    "Persistence preserves unavailable baseline-incumbent opportunities rather than filling them with zero rates.",
    persistence_group_summary.loc[
        persistence_group_summary["baseline_incumbent_opportunity_count"].eq(0),
        "incumbent_retention_rate",
    ].isna().all(),
    "No-opportunity retention rates remain unavailable",
    "Do not treat missing incumbent opportunity as zero persistence.",
)

add_check(
    "S4B010",
    "persistence_scope",
    "Persistence synthesis retains the four Stage 4A eligible records.",
    len(persistence) == 4,
    f"{len(persistence)} eligible persistence records",
    "Do not broaden persistence beyond eligible complete records.",
    caveat=True,
)

add_check(
    "S4B011",
    "mayora_sensitivity",
    "The Mayora extended structural sensitivity adds exactly one Le Minerale brand-family key.",
    (
        len(le_minerale_additional_keys) == 1
        and mayora_extended_count == mayora_primary_count + 1
    ),
    f"Mayora {mayora_primary_count} -> {mayora_extended_count} brand families",
    "Keep primary strict-control attribution unchanged.",
)

add_check(
    "S4B012",
    "mayora_sensitivity_rank",
    "Le Minerale sensitivity does not change Mayora structural breadth rank or the structural leader.",
    (
        mayora_extended_rank == mayora_primary_rank
        and extended_leaders == primary_leaders
    ),
    (
        f"Mayora rank {mayora_primary_rank} -> {mayora_extended_rank}; "
        f"leader(s)={' | '.join(primary_leaders)}"
    ),
    "Report the extended view only as sensitivity.",
)

add_check(
    "S4B013",
    "competitive_sensitivity",
    "Le Minerale sensitivity observations remain ineligible for quantitative group-performance comparison.",
    le_minerale_performance_candidates == 0,
    f"{len(le_minerale_competitive)} context observations; 0 performance candidates",
    "Do not mix context-only Brand Footprint facts into Top Brand analysis.",
)

add_check(
    "S4B014",
    "temporal_robustness",
    "Current and latest-historical complete focal leadership use the same category set.",
    same_category_set,
    f"same_category_set={same_category_set}",
    "If category sets differ, do not compare aggregate leader counts directly.",
)

add_check(
    "S4B015",
    "group_matrix",
    "Every focal group has all seven group-level evidence perspectives.",
    (
        len(group_dimension_evidence_matrix) == len(FOCAL_GROUPS) * 7
        and set(group_dimension_evidence_matrix["canonical_group"]) == set(FOCAL_GROUPS)
    ),
    f"{len(group_dimension_evidence_matrix)} group-dimension rows",
    "Preserve non-comparable dimensions outside the numeric matrix.",
)

add_check(
    "S4B016",
    "dimension_divergence",
    "Rankable dimensions are evaluated independently rather than forced to share one leader.",
    len(unique_rankable_leader_sets) >= 2,
    f"{len(unique_rankable_leader_sets)} distinct leader sets",
    "Report dimension-specific leaders.",
    caveat=True,
)

add_check(
    "S4B017",
    "consumer_reach",
    "Consumer reach remains non-comparable.",
    not consumer_reach_available,
    "not_comparable",
    "Do not infer CRP from rank, lower bounds, or qualitative facts.",
    caveat=True,
)

add_check(
    "S4B018",
    "concentration",
    "Portfolio concentration remains ineligible.",
    not concentration_available,
    "not_eligible",
    "Do not create performance HHI.",
)

add_check(
    "S4B019",
    "overall_winner",
    "Overall-winner defensibility is determined by explicit gates rather than a composite score.",
    not overall_winner_defensible,
    "not_defensible",
    "Use dimension-level reporting while blocking gates remain.",
    caveat=True,
)

forbidden_columns = {
    "composite_score",
    "overall_score",
    "weighted_score",
    "performance_hhi",
}

result_frames = {
    "leadership_snapshot_summary": leadership_snapshot_summary,
    "comparable_longitudinal_block_summary": comparable_longitudinal_block_summary,
    "persistence_group_summary": persistence_group_summary,
    "sensitivity_robustness_results": sensitivity_robustness_results,
    "group_dimension_evidence_matrix": group_dimension_evidence_matrix,
    "dimension_leader_summary": dimension_leader_summary,
    "overall_winner_defensibility": overall_winner_defensibility,
}

found_forbidden = {
    name: sorted(forbidden_columns & set(frame.columns))
    for name, frame in result_frames.items()
    if forbidden_columns & set(frame.columns)
}

add_check(
    "S4B020",
    "prohibited_synthesis",
    "No composite, weighted overall score, or performance HHI is produced.",
    not found_forbidden,
    "No prohibited synthesis fields",
    "Keep synthesis dimension-level and non-composite.",
)

stage4b_validation = pd.DataFrame(validation_rows)

if stage4b_validation["status"].eq("failed").any():
    final_gate = "FAIL"
else:
    final_gate = (
        "PASS_WITH_CAVEAT"
        if stage4b_validation["status"].eq("passed_with_caveat").any()
        else "PASS"
    )

gate_row = pd.DataFrame(
    [
        {
            "check_id": "S4B021",
            "validation_area": "stage_gate",
            "check_description": (
                "Stage 4B group-level comparative synthesis and sensitivity "
                "is complete enough for visualization and findings preparation."
            ),
            "result": final_gate,
            "status": (
                "passed_with_caveat"
                if final_gate == "PASS_WITH_CAVEAT"
                else "passed"
                if final_gate == "PASS"
                else "failed"
            ),
            "critical_failure": "yes" if final_gate == "FAIL" else "no",
            "required_treatment": (
                "Use dimension-specific evidence and retain the "
                "overall-winner-not-defensible conclusion unless later validated "
                "evidence resolves the blocking gates."
            ),
        }
    ]
)

stage4b_validation = pd.concat(
    [stage4b_validation, gate_row],
    ignore_index=True,
)

print(
    stage4b_validation[
        ["check_id", "validation_area", "result", "status"]
    ].to_string(index=False)
)
print(f"Final gate: {final_gate}")

if final_gate == "FAIL":
    failed_checks = stage4b_validation.loc[
        stage4b_validation["status"].eq("failed"),
        "check_id",
    ].tolist()
    raise RuntimeError(f"Stage 4B validation failed: {failed_checks}")


check_id                validation_area                                            result             status
  S4B001                input_integrity                       11/11 inputs passed SHA-256             passed
  S4B002               prior_stage_gate                                  PASS_WITH_CAVEAT passed_with_caveat
  S4B003    current_leadership_snapshot                     5 current complete categories             passed
  S4B004 historical_leadership_snapshot                   5 historical category snapshots passed_with_caveat
  S4B005           leadership_semantics         Selected-category rate semantics retained             passed
  S4B006             consistency_blocks                     5 comparable stability blocks passed_with_caveat
  S4B007                momentum_blocks                      5 comparable momentum blocks passed_with_caveat
  S4B008          longitudinal_boundary                  No 2025-2026 longitudinal bridge             passed
  S4B009      persi

## Deterministic Output Writing and Manifest

Write only the Stage 4B analytical synthesis, sensitivity, defensibility, and validation tables. The notebook performs no Git staging, commit, push, archive, or publishing operation.


In [12]:
OUTPUT_SPECS = [
    (
        "data/analytical/leadership_snapshot_summary.csv",
        leadership_snapshot_summary,
        (
            "Current 2026 and latest-historical complete focal-category "
            "leadership summaries by group"
        ),
    ),
    (
        "data/analytical/comparable_longitudinal_block_summary.csv",
        comparable_longitudinal_block_summary,
        (
            "Directly comparable category-period blocks for observed stability "
            "and momentum"
        ),
    ),
    (
        "data/analytical/persistence_group_summary.csv",
        persistence_group_summary,
        (
            "Group-level incumbent-retention opportunities and challenger "
            "takeover signals"
        ),
    ),
    (
        "data/analytical/sensitivity_robustness_results.csv",
        sensitivity_robustness_results,
        (
            "Pre-specified Mayora ownership sensitivity and temporal leadership "
            "robustness checks"
        ),
    ),
    (
        "data/analytical/group_dimension_evidence_matrix.csv",
        group_dimension_evidence_matrix,
        (
            "Non-composite group-by-dimension comparative evidence matrix"
        ),
    ),
    (
        "data/analytical/dimension_leader_summary.csv",
        dimension_leader_summary,
        (
            "Dimension-specific leaders and non-rankable descriptive signals"
        ),
    ),
    (
        "data/analytical/overall_winner_defensibility.csv",
        overall_winner_defensibility,
        (
            "Explicit synthesis gates assessing whether one overall portfolio "
            "winner is defensible"
        ),
    ),
    (
        "metadata/stage4b_analysis_validation.csv",
        stage4b_validation,
        "Stage 4B validation checks and final gate",
    ),
]

for relative_path, frame, _ in OUTPUT_SPECS:
    destination = OUTPUT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(
        destination,
        index=False,
        lineterminator="\n",
    )

manifest_rows = []

for relative_path, frame, description in OUTPUT_SPECS:
    path = OUTPUT_ROOT / relative_path
    manifest_rows.append(
        {
            "file_path": relative_path,
            "row_count": len(frame),
            "sha256": sha256_file(path),
            "description": description,
            "repository_head": REPOSITORY_HEAD,
        }
    )

stage4b_manifest = pd.DataFrame(manifest_rows)
manifest_path = METADATA_ROOT / "stage4b_output_manifest.csv"
stage4b_manifest.to_csv(
    manifest_path,
    index=False,
    lineterminator="\n",
)

written_paths = [
    OUTPUT_ROOT / relative_path
    for relative_path, _, _ in OUTPUT_SPECS
] + [manifest_path]

if not all(path.exists() for path in written_paths):
    raise FileNotFoundError("At least one Stage 4B output was not written.")

print(stage4b_manifest.to_string(index=False))
print()
print(f"Stage 4B final gate: {final_gate}")
print(f"Overall winner defensible: {overall_winner_defensible}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Output files written: {len(written_paths)}")


                                                file_path  row_count                                                           sha256                                                                              description                          repository_head
          data/analytical/leadership_snapshot_summary.csv          8 eedf17e9f397bcc5e30ed5bca20757534930e21ad74844a3e866211765924d4e Current 2026 and latest-historical complete focal-category leadership summaries by group 6ee32d06e296b8e06f1976fb7ba1bc3dcbe5f935
data/analytical/comparable_longitudinal_block_summary.csv         10 6064a3d6d65eeaa568cd56086127e4a00fc04d11707349aed35f7f1958bc476c           Directly comparable category-period blocks for observed stability and momentum 6ee32d06e296b8e06f1976fb7ba1bc3dcbe5f935
            data/analytical/persistence_group_summary.csv          4 8835756aa59c38d60888d9d08eaad4a17b7a42c2f1c49581a634299bcc8685a7            Group-level incumbent-retention opportunities and challenger ta